# Bâtis de sociétaires impactés par les incendies — Gironde & Biscarrosse

**Demande métier** — *« Je souhaiterais savoir combien et positionner géographiquement
les bâtis (RP, RS, PNO) de nos sociétaires qui auraient effectivement été impactés
par les incendies. »*

**Livrable** — un fichier HTML autoportant (`livrables/carte_incendie_societaires.html`)
contenant les compteurs, la carte interactive et les tableaux, plus un export CSV
des contrats impactés pour les équipes de gestion.

---

## Méthode en une phrase

On croise les **points GPS des contrats habitation** (`contrat_mgar_gps_iris`) avec les
**emprises des bâtiments relevés dans la zone brûlée** par la cellule SIG, et on classe
chaque contrat par **niveau de certitude d'impact** selon sa distance au bâti le plus proche.

## Ce que contiennent les données incendie (analyse préalable)

| Couche | Fichier | Nb | CRS déclaré | Aire recalculée | Attributs |
|---|---|---|---|---|---|
| Contour feu Gironde | `2026_07_26_16h_Contour feu Gironde.shp` | 1 polygone | **EPSG:4326** | **37 020 ha** | `Surface` = 9702, `sup 2607` = 38502 |
| Bâti concerné Gironde | `2026_07_26_Bati_concerné_feu_Gironde.shp` | **1 607** | EPSG:2154 | 28,5 ha bâtis | `cleabs` *(inexploitable)*, `Surface` (m²) |
| Contour feu Biscarrosse | `2026_07_26_16hContour feu Biscarosse.shp` | 1 polygone | **absent (.prj manquant)** | **3 422 ha** | `FID` |
| Bâti concerné Biscarrosse | `2026_07_2026_Bati_concerné_feu_Biscarrosse.shp` | **1 060** | EPSG:2154 | 22,9 ha bâtis | `Surface` (m²) |

Points de vigilance relevés et traités dans ce notebook :

1. **CRS hétérogènes.** Les contours sont en WGS84 (Gironde) et sans projection déclarée
   (Biscarrosse). Les emprises de Biscarrosse sont bien du **Lambert 93** — vérifié par
   l'étendue des coordonnées (x ≈ 368 000, y ≈ 6 372 000) et par le recouvrement à 98 %
   avec les bâtis du même dossier. Le notebook force donc `EPSG:2154` pour ce fichier.
2. **`cleabs` est inexploitable** sur le fichier Gironde : la même valeur
   `BATIMENT0000000245412688` est répétée sur les 1 607 lignes (artefact de jointure).
   On génère donc notre propre identifiant de bâtiment (`bat_id`).
3. **Sur les couches de bâti, `Surface` est bien l'emprise au sol** en m² (contrôlé :
   `Surface` == aire géométrique calculée en Lambert 93). Minimum 50 m² sur les deux
   couches → les petites annexes (abris, cabanes) ont déjà été filtrées à la source.
   En revanche, sur le **contour** Gironde les attributs `Surface` (9 702) et
   `sup 2607` (38 502) ne concordent pas avec l'aire géométrique (37 020 ha) : ces
   deux champs ne sont pas repris, toutes les surfaces du notebook sont **recalculées**.
4. **Périmètre ≠ bâti concerné.** Le contour du feu couvre 370 km² (Gironde) et 34 km²
   (Biscarrosse), en grande majorité de la forêt. Répondre « impactés » par
   *point dans le périmètre* surestimerait massivement : la référence à retenir est le
   **bâti relevé dans l'emprise**, le périmètre ne servant qu'à qualifier l'exposition.

## 1. Paramètres

Tout ce qui se règle est ici. Les seuils de distance sont le seul vrai arbitrage :
ils absorbent l'imprécision du géocodage des adresses.

In [ ]:
from __future__ import annotations

import html as _html
import json
import warnings
from datetime import date
from pathlib import Path

import folium
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# --- Arborescence ---------------------------------------------------------- #
RACINE = Path.cwd()
if RACINE.name == "incendie":                     # notebook lancé depuis son dossier
    RACINE = RACINE.parent
DOSSIER_INCENDIE = RACINE / "data_incendie"
DOSSIER_SORTIE = RACINE / "livrables"
DOSSIER_SORTIE.mkdir(exist_ok=True)

FICHIER_HTML = DOSSIER_SORTIE / "carte_incendie_societaires.html"
FICHIER_CSV = DOSSIER_SORTIE / "contrats_impactes.csv"

# --- Les deux sinistres ---------------------------------------------------- #
# `crs_defaut` ne sert que si le shapefile n'embarque pas de .prj.
FEUX = {
    "Gironde":     {"dossier": "FEU GIRONDE",    "contour": "*Contour*.shp",
                    "bati": "*Bati*.shp", "crs_defaut": 2154},
    "Biscarrosse": {"dossier": "FEU BISCAROSSE", "contour": "*Contour*.shp",
                    "bati": "*Bati*.shp", "crs_defaut": 2154},
}

# --- Géodésie -------------------------------------------------------------- #
CRS_METRIQUE = 2154        # Lambert 93 : toutes les distances sont calculées ici
CRS_AFFICHAGE = 4326       # WGS84 : projection de la carte

# --- Seuils d'appariement (mètres) ----------------------------------------- #
# Un point contrat tombe rarement pile dans l'emprise du bâtiment : le géocodage
# ramène souvent l'adresse au bord de la parcelle ou à l'axe de la voie.
SEUIL_TRES_PROBABLE_M = 10.0    # au-delà de l'emprise mais collé à un bâti de l'emprise
SEUIL_PROBABLE_M = 25.0         # tolérance haute du géocodage adresse
SEUILS_SENSIBILITE_M = [0.0, 5.0, 10.0, 25.0, 50.0, 100.0]

# Niveaux retenus comme « effectivement impacté » dans les compteurs de tête.
NIVEAUX_IMPACTES = ("Certain — dans l'emprise brûlée", "Très probable — à moins de 10 m", "Probable — à moins de 25 m")

# --- Segmentation demandée ------------------------------------------------- #
SEGMENTS_CIBLE = ("RP", "RS", "PNO")

# --- Carte ----------------------------------------------------------------- #
FOND_DE_CARTE = "OpenStreetMap"   # None => aucun fond tuilé (poste sans accès Internet)
# Leaflet est embarqué dans le dépôt (incendie/assets) et injecté en dur dans le HTML :
# le livrable ne dépend donc d'aucun CDN, seules les tuiles du fond de carte
# nécessitent un accès réseau. Mettre False pour repasser sur les CDN de folium.
DOSSIER_ASSETS = RACINE / "incendie" / "assets"
LEAFLET_EMBARQUE = all((DOSSIER_ASSETS / f).exists()
                       for f in ("leaflet.js", "leaflet.css", "jquery.js"))

DATE_ANALYSE = date.today().isoformat()
print("Racine projet :", RACINE)
print("Données incendie :", DOSSIER_INCENDIE, "—", DOSSIER_INCENDIE.exists())

## 2. Chargement et normalisation des données incendie

On ramène tout en Lambert 93, on force le CRS manquant de Biscarrosse et on
crée un identifiant de bâtiment fiable.

In [ ]:
def _lire_couche(chemin: Path, crs_defaut: int) -> gpd.GeoDataFrame:
    """Lit un shapefile et le ramène en Lambert 93, en réparant un CRS absent."""
    gdf = gpd.read_file(chemin)
    if gdf.crs is None:
        gdf = gdf.set_crs(crs_defaut)
        print(f"  ⚠️  {chemin.name} : aucun .prj → CRS forcé à EPSG:{crs_defaut}")
    return gdf.to_crs(CRS_METRIQUE)


contours_l, batis_l = [], []
for nom, cfg in FEUX.items():
    dossier = DOSSIER_INCENDIE / cfg["dossier"]
    f_contour = sorted(dossier.glob(cfg["contour"]))[0]
    f_bati = sorted(dossier.glob(cfg["bati"]))[0]
    print(f"• {nom}")

    contour = _lire_couche(f_contour, cfg["crs_defaut"])
    contour["feu"] = nom
    contour["fichier"] = f_contour.name
    contours_l.append(contour[["feu", "fichier", "geometry"]])

    bati = _lire_couche(f_bati, cfg["crs_defaut"])
    bati["feu"] = nom
    bati["fichier"] = f_bati.name
    bati["surface_bati_m2"] = bati.geometry.area.round(1)
    batis_l.append(bati[["feu", "fichier", "surface_bati_m2", "geometry"]])

contours = gpd.GeoDataFrame(pd.concat(contours_l, ignore_index=True),
                            geometry="geometry", crs=CRS_METRIQUE)
batis = gpd.GeoDataFrame(pd.concat(batis_l, ignore_index=True),
                         geometry="geometry", crs=CRS_METRIQUE)

# `cleabs` étant constant sur le fichier Gironde, on fabrique notre propre clé.
batis["bat_id"] = [f"{f[:4].upper()}-{i:05d}" for i, f in enumerate(batis["feu"], 1)]

contours["surface_feu_ha"] = (contours.geometry.area / 10_000).round(0)

print()
display(contours.drop(columns="geometry"))
display(
    batis.groupby("feu")
    .agg(nb_batis=("bat_id", "size"),
         surface_totale_m2=("surface_bati_m2", "sum"),
         surface_mediane_m2=("surface_bati_m2", "median"),
         surface_max_m2=("surface_bati_m2", "max"))
    .round(0)
)

In [ ]:
# Contrôle de cohérence : les bâtis relevés tombent-ils bien dans leur périmètre ?
for nom in FEUX:
    c = contours.loc[contours.feu == nom].geometry.union_all()
    b = batis.loc[batis.feu == nom]
    dedans = b.representative_point().within(c).mean()
    print(f"{nom:<12} {len(b):>5} bâtis — {dedans:6.1%} strictement dans le contour "
          f"(le reste affleure la limite du périmètre)")

## 3. Extraction des contrats habitation

On reprend la requête fournie — jointure `contrat_mgar_gps_iris` × `contrat_mgar` sur
(`id_societaire`, `numero_intercalaire`, `code_postal_adresse_risque`), filtrée sur
`tech_date_fin_historisation IS NULL`, avec `id = CONCAT(id_societaire, numero_intercalaire)`
— complétée par :

* un **pré-filtre sur la bounding box des deux feux**, calculée depuis les shapefiles :
  inutile de rapatrier la France entière pour une analyse landaise/girondine ;
* quelques **colonnes facultatives** (`code_type_bien`, `commune_adresse_risque`,
  `surface_habitable`…) qui enrichissent le livrable. Le notebook fonctionne sans :
  il détecte les colonnes présentes et adapte tableaux et export.

`t1.geom` est écarté après chargement : il fait doublon avec `lon_contrat_mgar` /
`lat_contrat_mgar`, à partir desquels la géométrie est reconstruite côté Python.

Le notebook s'exécute dans **trois modes**, dans cet ordre :
`BigQuery` → `fichier local` → `simulation` (jeu de test synthétique, pour valider
la chaîne de bout en bout sans accès aux données réelles).

In [ ]:
# Emprise des deux feux, élargie de 2 km, en WGS84 -> pré-filtre de la requête.
bbox = (contours.geometry.buffer(2_000).to_crs(CRS_AFFICHAGE).total_bounds)
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = [round(v, 4) for v in bbox]
print(f"Bounding box de travail : lon [{LON_MIN}, {LON_MAX}] — lat [{LAT_MIN}, {LAT_MAX}]")

# Colonnes facultatives : elles enrichissent le livrable (nature du bien, commune,
# surface) et sont détectées automatiquement plus bas. Mettre "" pour s'en passer.
COLONNES_OPTIONNELLES = """,
  t2.code_type_bien,                  -- IM appartement / MP maison / MH mobile home
  t2.commune_adresse_risque,
  t2.surface_habitable,
  t2.nombre_pieces_totales,
  t2.montant_capital_mobilier"""

REQUETE_SQL = f"""
SELECT
  t1.id_societaire,
  t1.numero_intercalaire,
  CONCAT(t1.id_societaire, t1.numero_intercalaire) AS id,
  t1.rue_adresse_risque,
  t1.code_postal_adresse_risque,
  t1.lon_contrat_mgar,
  t1.lat_contrat_mgar,
  t1.geom,
  t2.code_sous_type,                  -- segmentation RP / RS / PNO
  t2.code_qualite_assure_habitation{COLONNES_OPTIONNELLES}
FROM `matmut-dda-irisation-prd-26f3.gold_irisation_prd.contrat_mgar_gps_iris` t1
INNER JOIN `matmut-dni-datalake-prd-8ec7.silver_produitetcontrat_sigmainframe_prd.contrat_mgar` t2
  ON  t1.id_societaire              = t2.id_societaire
  AND t1.numero_intercalaire        = t2.numero_intercalaire
  AND t1.code_postal_adresse_risque = t2.code_postal_adresse_risque
WHERE t2.tech_date_fin_historisation IS NULL
  -- Pré-filtre sur l'emprise des deux feux : évite de rapatrier la France entière
  AND t1.lon_contrat_mgar BETWEEN {LON_MIN} AND {LON_MAX}
  AND t1.lat_contrat_mgar BETWEEN {LAT_MIN} AND {LAT_MAX}
"""
print(REQUETE_SQL)

In [ ]:
PROJET_BQ = "matmut-dda-irisation-prd-26f3"
FICHIER_LOCAL = DOSSIER_INCENDIE / "export_societaires.csv"   # export manuel éventuel

COLONNES_MIN = ["id", "id_societaire", "numero_intercalaire",
                "lon_contrat_mgar", "lat_contrat_mgar", "code_sous_type"]


def charger_contrats() -> tuple[pd.DataFrame, str]:
    """BigQuery, sinon export local, sinon jeu simulé. Renvoie (df, mode)."""
    try:
        from google.cloud import bigquery
        df = bigquery.Client(project=PROJET_BQ).query(REQUETE_SQL).to_dataframe()
        return df, "bigquery"
    except Exception as exc:                                    # noqa: BLE001
        print(f"BigQuery indisponible ({type(exc).__name__}: {exc}).")

    if FICHIER_LOCAL.exists():
        df = pd.read_csv(FICHIER_LOCAL, dtype=str)
        for c in ("lon_contrat_mgar", "lat_contrat_mgar"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
        return df, "fichier"

    print(f"Aucun export dans {FICHIER_LOCAL} → génération d'un jeu SIMULÉ.")
    return simuler_contrats(), "simulation"


def simuler_contrats(n_sur_bati: int = 180, n_perimetre: int = 260,
                     n_alentours: int = 900, graine: int = 20260727) -> pd.DataFrame:
    """Jeu de test synthétique : permet de dérouler tout le notebook sans données réelles."""
    rng = np.random.default_rng(graine)
    pts = []

    # a) contrats posés sur des bâtis relevés dans l'emprise brûlée
    ech = batis.sample(n_sur_bati, random_state=graine)
    for geom in ech.representative_point():
        d = rng.normal(0, 9, 2)
        pts.append((geom.x + d[0], geom.y + d[1]))

    # b) contrats dans le périmètre mais loin de tout bâti relevé
    for _ in range(n_perimetre):
        x0, y0, x1, y1 = contours.sample(1, random_state=int(rng.integers(1e6))).total_bounds
        pts.append((rng.uniform(x0, x1), rng.uniform(y0, y1)))

    # c) contrats des communes alentour, hors périmètre
    x0, y0, x1, y1 = contours.total_bounds
    for _ in range(n_alentours):
        pts.append((rng.uniform(x0 - 12_000, x1 + 12_000),
                    rng.uniform(y0 - 12_000, y1 + 12_000)))

    g = gpd.GeoSeries([Point(x, y) for x, y in pts], crs=CRS_METRIQUE).to_crs(CRS_AFFICHAGE)
    n = len(g)
    sous_types = rng.choice(list("1234567") + ["8", "J", "A", "H"], n,
                            p=[.10, .12, .30, .07, .05, .18, .10, .03, .02, .02, .01])
    soc = [f"SIM{i:09d}" for i in range(n)]
    inter = rng.choice(["80", "81", "82"], n)
    return pd.DataFrame({
        "id_societaire": soc,
        "numero_intercalaire": inter,
        "id": [s + i for s, i in zip(soc, inter)],
        "rue_adresse_risque": "ADRESSE SIMULEE",
        "code_postal_adresse_risque": rng.choice(["33990", "40600", "33680"], n),
        "commune_adresse_risque": rng.choice(["LACANAU", "BISCARROSSE", "LE PORGE"], n),
        "lon_contrat_mgar": g.x.values,
        "lat_contrat_mgar": g.y.values,
        "code_sous_type": sous_types,
        "code_qualite_assure_habitation": rng.choice(["P", "L", "H"], n, p=[.60, .36, .04]),
        "code_type_bien": rng.choice(["MP", "IM", "MH"], n, p=[.62, .33, .05]),
        "surface_habitable": rng.integers(35, 180, n).astype(str),
        "nombre_pieces_totales": rng.choice(["02", "03", "04", "05"], n),
        "montant_capital_mobilier": rng.choice([10_000, 15_000, 25_000, 35_000], n),
    })


contrats, MODE_SOURCE = charger_contrats()
contrats = contrats.drop(columns=["geom"], errors="ignore")   # WKT redondant avec lon/lat
manquantes = [c for c in COLONNES_MIN if c not in contrats.columns]
assert not manquantes, f"Colonnes absentes de l'export : {manquantes}"

# `id` = CONCAT(id_societaire, numero_intercalaire) : la jointure portant aussi sur le
# code postal, un même `id` peut théoriquement revenir sur deux adresses. On vérifie.
n_dup = int(contrats["id"].duplicated().sum())
print(f"\nMode = {MODE_SOURCE.upper()} — {len(contrats):,} lignes chargées".replace(",", " "))
print(f"{contrats['id'].nunique():,} identifiants contrat distincts".replace(",", " ")
      + (f" — ⚠️ {n_dup} doublon(s) d'`id` (même sociétaire/intercalaire, code postal "
         "différent) : les comptages dédoublonnent sur `id`." if n_dup else ""))
print("Colonnes facultatives présentes :",
      [c for c in ("code_type_bien", "commune_adresse_risque", "surface_habitable",
                   "nombre_pieces_totales", "montant_capital_mobilier")
       if c in contrats.columns] or "aucune")
contrats.head()

## 4. Segmentation RP / RS / PNO

Le type de contrat se lit sur **`code_sous_type`** (source
`cd_opt_mgar_of_donnees_sas_mgar_of_cwhymgaz`), regroupement fonctionnel documenté
dans le dictionnaire `CONTRAT_MGAR` :

| `code_sous_type` | Segment |
|---|---|
| `1` `2` `3` `4` `5` | **RP** — résidence principale |
| `6` | **PNO** — propriétaire non occupant |
| `7` | **RS** — résidence secondaire |
| `8` `9` `J` | JEUN — contrat jeune |
| `A` `B` `C` `D` `P` `R` | ETUD — étudiant |
| `E` `F` | ETUE — étudiant étranger |
| `H` | HEB — hébergé |
| `T` | TBNH — temporairement bien non habité |

La demande porte sur **RP / RS / PNO**. Les autres segments sont malgré tout comptés
et affichés à part : ce sont aussi des logements occupés, et les exclure sans le dire
minorerait le nombre de sociétaires touchés.

In [ ]:
MAP_SOUS_TYPE = {
    **{c: "RP" for c in "12345"},
    "6": "PNO",
    "7": "RS",
    **{c: "JEUN" for c in "89J"},
    **{c: "ETUD" for c in "ABCDPR"},
    **{c: "ETUE" for c in "EF"},
    "H": "HEB",
    "T": "TBNH",
}
LIB_SEGMENT = {
    "RP": "Résidence principale", "RS": "Résidence secondaire",
    "PNO": "Propriétaire non occupant", "JEUN": "Contrat jeune",
    "ETUD": "Étudiant", "ETUE": "Étudiant étranger",
    "HEB": "Hébergé", "TBNH": "Bien temporairement non habité",
    "INCONNU": "Sous-type absent ou non référencé",
}
LIB_TYPE_BIEN = {"MP": "Maison particulière", "IM": "Appartement", "MH": "Mobile home"}
LIB_QUALITE = {"P": "Propriétaire", "L": "Locataire", "H": "Hébergé gratuit",
               "I": "Colocation individuelle", "R": "Chambre maison de retraite",
               "M": "Chambre établissement médical", "G": "Colocation commune",
               "N": "Nu-propriétaire", "C": "Logement de service",
               "U": "Usufruitier", "S": "Sans résidence fixe"}

contrats["code_sous_type"] = contrats["code_sous_type"].astype("string").str.strip().str.upper()
contrats["segment"] = contrats["code_sous_type"].map(MAP_SOUS_TYPE).fillna("INCONNU")
contrats["dans_perimetre_demande"] = contrats["segment"].isin(SEGMENTS_CIBLE)
if "code_type_bien" in contrats:
    contrats["type_bien"] = contrats["code_type_bien"].map(LIB_TYPE_BIEN).fillna("Non renseigné")
if "code_qualite_assure_habitation" in contrats:
    contrats["qualite"] = (contrats["code_qualite_assure_habitation"].astype("string")
                           .str.strip().str.upper().map(LIB_QUALITE).fillna("Non renseigné"))

repartition = (contrats["segment"].value_counts(dropna=False).rename("contrats")
               .to_frame()
               .assign(part=lambda d: (d.contrats / d.contrats.sum()).map("{:.1%}".format),
                       libelle=lambda d: d.index.map(LIB_SEGMENT),
                       demande=lambda d: np.where(d.index.isin(SEGMENTS_CIBLE), "✅", "—")))
display(repartition[["libelle", "contrats", "part", "demande"]])

## 5. Qualité du géocodage

Étape indispensable avant de compter : si une partie des contrats est géocodée au
**centroïde de la commune** plutôt qu'à l'adresse, tout point tombant par hasard près
d'un bâti de l'emprise produirait un faux positif. On repère ces coordonnées
sur-représentées et on les signale.

In [ ]:
contrats = contrats.dropna(subset=["lon_contrat_mgar", "lat_contrat_mgar"]).copy()

pts = gpd.GeoDataFrame(
    contrats,
    geometry=gpd.points_from_xy(contrats["lon_contrat_mgar"], contrats["lat_contrat_mgar"]),
    crs=CRS_AFFICHAGE,
).to_crs(CRS_METRIQUE)

cle_xy = (pts.geometry.x.round(0).astype(int).astype(str) + "_"
          + pts.geometry.y.round(0).astype(int).astype(str))
doublons = cle_xy.value_counts()
suspects = doublons[doublons >= 5]

pts["geocodage_suspect"] = cle_xy.isin(suspects.index).values
print(f"{len(pts):,} contrats géolocalisés".replace(",", " "))
print(f"{len(suspects)} coordonnées portant ≥ 5 contrats "
      f"({pts['geocodage_suspect'].sum()} contrats, {pts['geocodage_suspect'].mean():.1%}) "
      f"→ probables centroïdes commune/voie, à confirmer avant toute action terrain.")

## 6. Appariement spatial

Pour chaque contrat, on cherche le **bâtiment le plus proche relevé dans l'emprise brûlée** puis on classe :

| Niveau | Règle | Lecture |
|---|---|---|
| **Certain — dans l'emprise brûlée** | le point tombe **dans** l'emprise d'un bâti brûlé | le logement assuré est l'un des bâtiments de l'emprise |
| **Très probable — à moins de 10 m** | ≤ 10 m d'un bâti brûlé | décalage de géocodage courant (bord de parcelle) |
| **Probable — à moins de 25 m** | ≤ 25 m d'un bâti brûlé | tolérance haute du géocodage adresse |
| **Exposé — dans le périmètre du feu** | dans le contour, > 25 m de tout bâti brûlé | exposé, bâti non relevé dans l'emprise |
| **Hors périmètre** | reste | non concerné |

Les trois premiers niveaux constituent la réponse à « **effectivement impactés** ».
Le tableau de sensibilité plus bas montre l'effet du seuil sur le compte.

In [ ]:
BAT_COLS = ["bat_id", "feu", "surface_bati_m2", "geometry"]

# Bâti relevé le plus proche, dans la limite du seuil le plus large
appar = gpd.sjoin_nearest(
    pts, batis[BAT_COLS], how="left",
    max_distance=max(SEUILS_SENSIBILITE_M), distance_col="distance_bati_m",
)
# sjoin_nearest peut renvoyer plusieurs ex-aequo : on garde le plus proche
appar = (appar.sort_values("distance_bati_m")
              .loc[~appar.index.duplicated(keep="first")]
              .sort_index()
              .drop(columns=["index_right"], errors="ignore"))

# Appartenance au périmètre du feu
peri = gpd.sjoin(pts[["geometry"]], contours[["feu", "geometry"]],
                 how="left", predicate="within")
peri = peri.loc[~peri.index.duplicated(keep="first")]
appar["feu_perimetre"] = peri["feu"]


def niveau(r):
    d = r["distance_bati_m"]
    if pd.notna(d):
        if d <= 0:
            return "Certain — dans l'emprise brûlée"
        if d <= SEUIL_TRES_PROBABLE_M:
            return "Très probable — à moins de 10 m"
        if d <= SEUIL_PROBABLE_M:
            return "Probable — à moins de 25 m"
    if pd.notna(r["feu_perimetre"]):
        return "Exposé — dans le périmètre du feu"
    return "Hors périmètre"


appar["niveau_impact"] = appar.apply(niveau, axis=1)
appar["feu_rattache"] = appar["feu"].fillna(appar["feu_perimetre"])
appar["est_impacte"] = appar["niveau_impact"].isin(NIVEAUX_IMPACTES)
# On garde le bâti le plus proche quel que soit le seuil (utile au test de sensibilité),
# mais un bâtiment n'est déclaré « touché » que pour les contrats effectivement impactés.
appar["bat_id_brut"] = appar["bat_id"]
appar.loc[~appar["est_impacte"], "bat_id"] = pd.NA

ORDRE_NIVEAUX = ["Certain — dans l'emprise brûlée", "Très probable — à moins de 10 m", "Probable — à moins de 25 m",
                 "Exposé — dans le périmètre du feu", "Hors périmètre"]
appar["niveau_impact"] = pd.Categorical(appar["niveau_impact"], ORDRE_NIVEAUX, ordered=True)

display(appar["niveau_impact"].value_counts().sort_index().rename("contrats").to_frame())

## 7. Réponse chiffrée

« Combien ? » se décline en trois compteurs différents, et **il faut les trois** :

* **bâtiments distincts** touchés portant au moins un contrat — la réponse littérale
  à la question, dédoublonnée (un immeuble ne compte qu'une fois) ;
* **contrats** impactés — la charge de gestion réelle ;
* **sociétaires distincts** — le nombre de foyers à contacter.

In [ ]:
cible = appar[appar["dans_perimetre_demande"]]          # RP / RS / PNO
impact = cible[cible["est_impacte"]]

KPI = {
    "batis_touches": int(impact["bat_id"].nunique()),
    "contrats": int(impact["id"].nunique()),
    "societaires": int(impact["id_societaire"].nunique()),
    "certains": int((impact["niveau_impact"] == "Certain — dans l'emprise brûlée").sum()),
    "en_perimetre": int((cible["niveau_impact"] == "Exposé — dans le périmètre du feu").sum()),
    "hors_cible_impactes": int(appar.loc[~appar["dans_perimetre_demande"]
                                         & appar["est_impacte"], "id"].nunique()),
    "suspects": int(impact["geocodage_suspect"].sum()),
}
for k, v in KPI.items():
    print(f"{k:>22} : {v:>6,}".replace(",", " "))

In [ ]:
# --- Tableau 1 : synthèse par feu et par segment --------------------------- #
synthese = (impact.groupby(["feu_rattache", "segment"], observed=True)
                  .agg(batis_touches=("bat_id", "nunique"),
                       contrats=("id", "nunique"),
                       societaires=("id_societaire", "nunique"))
                  .reset_index()
                  .rename(columns={"feu_rattache": "feu"}))
synthese["libelle"] = synthese["segment"].map(LIB_SEGMENT)
synthese = synthese.sort_values(["feu", "contrats"], ascending=[True, False])
display(synthese[["feu", "segment", "libelle", "batis_touches", "contrats", "societaires"]])

# --- Tableau 2 : détail par niveau de certitude ---------------------------- #
detail = (pd.crosstab(cible["feu_rattache"].fillna("Aucun feu à proximité"),
                      cible["niveau_impact"], dropna=False)
            .reindex(columns=ORDRE_NIVEAUX, fill_value=0))
detail.index.name = "Feu le plus proche"
display(detail)

In [ ]:
# --- Tableau 3 : sensibilité au seuil de distance -------------------------- #
lignes = []
for s in SEUILS_SENSIBILITE_M:
    sel = cible[cible["distance_bati_m"].le(s)]
    # `bat_id` a été neutralisé au-delà du seuil retenu : on le relit sur l'appariement brut
    bat_s = appar.loc[sel.index, "bat_id_brut"]
    lignes.append({"seuil_m": s,
                   "batis_touches": int(bat_s.nunique()),
                   "contrats": int(sel["id"].nunique()),
                   "societaires": int(sel["id_societaire"].nunique())})
sensibilite = pd.DataFrame(lignes)
sensibilite["retenu"] = np.where(sensibilite["seuil_m"] == SEUIL_PROBABLE_M, "◀ retenu", "")
display(sensibilite)

In [ ]:
# --- Tableau 4 : qualité de l'assuré et nature du bien --------------------- #
if "qualite" in impact:
    display(pd.crosstab(impact["qualite"], impact["segment"],
                        margins=True, margins_name="Total"))
if "type_bien" in impact:
    display(pd.crosstab(impact["type_bien"], impact["segment"],
                        margins=True, margins_name="Total"))

# --- Export gestion --------------------------------------------------------- #
cols_export = [c for c in [
    "id", "id_societaire", "numero_intercalaire", "segment",
    "qualite", "type_bien", "rue_adresse_risque",
    "code_postal_adresse_risque", "commune_adresse_risque",
    "lon_contrat_mgar", "lat_contrat_mgar", "feu_rattache", "niveau_impact",
    "distance_bati_m", "bat_id", "surface_bati_m2", "surface_habitable",
    "nombre_pieces_totales", "montant_capital_mobilier", "geocodage_suspect",
] if c in impact.columns]

export = (impact[cols_export]
          .sort_values(["feu_rattache", "niveau_impact", "distance_bati_m"]))
export.to_csv(FICHIER_CSV, index=False, encoding="utf-8-sig")
print(f"{len(export)} contrats exportés → {FICHIER_CSV}")
export.head(10)

## 8. Carte interactive

Encodage : la **couleur porte le segment** (RP / RS / PNO — trois teintes validées
pour la lisibilité en vision des couleurs déficiente), la **taille et l'opacité
portent le niveau de certitude**. Chaque niveau est une couche activable
séparément, pour isoler les cas certains.

In [ ]:
PALETTE_SEGMENT = {"RP": "#2a78d6", "RS": "#eb6834", "PNO": "#1baf7a", "Autres": "#8a8983"}
STYLE_NIVEAU = {
    "Certain — dans l'emprise brûlée": dict(radius=7.5, fill_opacity=0.95, weight=2.0),
    "Très probable — à moins de 10 m":   dict(radius=6.0, fill_opacity=0.75, weight=1.5),
    "Probable — à moins de 25 m":        dict(radius=5.0, fill_opacity=0.50, weight=1.2),
    "Exposé — dans le périmètre du feu": dict(radius=3.5, fill_opacity=0.25, weight=0.8),
}

carte_pts = appar.to_crs(CRS_AFFICHAGE)
contours_wgs = contours.to_crs(CRS_AFFICHAGE)
centre = contours_wgs.geometry.union_all().centroid

m = folium.Map(location=[centre.y, centre.x], tiles=FOND_DE_CARTE, control_scale=True)
# Les deux feux sont distants de ~60 km : on cadre sur leur emprise commune
# plutôt que sur un niveau de zoom fixe, sinon Biscarrosse sort de l'écran.
x0, y0, x1, y1 = contours_wgs.total_bounds
m.fit_bounds([[y0, x0], [y1, x1]], padding=(20, 20))

# Périmètres des feux
folium.GeoJson(
    contours.to_crs(CRS_AFFICHAGE),
    name="Périmètre des feux",
    style_function=lambda _: {"color": "#e34948", "weight": 2.5,
                              "fillColor": "#e34948", "fillOpacity": 0.06},
    tooltip=folium.GeoJsonTooltip(fields=["feu", "surface_feu_ha"],
                                  aliases=["Feu", "Surface (ha)"]),
).add_to(m)

# Bâtiments relevés dans l'emprise brûlée
folium.GeoJson(
    batis.to_crs(CRS_AFFICHAGE),
    name=f"Bâtiments dans l'emprise brûlée ({len(batis)})",
    style_function=lambda _: {"color": "#0d366b", "weight": 0.6,
                              "fillColor": "#1f2937", "fillOpacity": 0.85},
    tooltip=folium.GeoJsonTooltip(fields=["feu", "bat_id", "surface_bati_m2"],
                                  aliases=["Feu", "Bâtiment", "Emprise (m²)"]),
).add_to(m)

# Contrats, une couche par niveau de certitude
for niveau_lbl, style in STYLE_NIVEAU.items():
    sel = carte_pts[(carte_pts["niveau_impact"] == niveau_lbl)
                    & carte_pts["dans_perimetre_demande"]]
    if sel.empty:
        continue
    fg = folium.FeatureGroup(name=f"{niveau_lbl} ({len(sel)})",
                             show=niveau_lbl in NIVEAUX_IMPACTES)
    for r in sel.itertuples():
        couleur = PALETTE_SEGMENT.get(r.segment, PALETTE_SEGMENT["Autres"])
        dist = "—" if pd.isna(r.distance_bati_m) else f"{r.distance_bati_m:.0f} m"
        adresse = " ".join(str(getattr(r, c, "") or "") for c in
                           ("rue_adresse_risque", "code_postal_adresse_risque",
                            "commune_adresse_risque")).strip()
        popup = (f"<b>{LIB_SEGMENT.get(r.segment, r.segment)}</b><br>"
                 f"Contrat : {r.id}<br>"
                 f"{adresse}<br>"
                 f"Niveau : {r.niveau_impact}<br>"
                 f"Distance au bâti de l'emprise : {dist}"
                 + ("<br><i>⚠️ géocodage à confirmer</i>" if r.geocodage_suspect else ""))
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            color="#ffffff", fillColor=couleur, fill=True,
            popup=folium.Popup(popup, max_width=320), **style,
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(position="topright", collapsed=False).add_to(m)

# Raccourcis de cadrage. À l'échelle qui couvre les deux feux (60 km d'écart), un
# bâtiment de 137 m² occupe moins d'un pixel : la vue d'ensemble sert à situer, pas
# à lire. Ces liens amènent directement à l'échelle où le bâti devient lisible.
zones = {"Vue d'ensemble": [[y0, x0], [y1, x1]]}
for nom_feu in FEUX:
    fx0, fy0, fx1, fy1 = contours_wgs.loc[contours_wgs.feu == nom_feu].total_bounds
    zones[f"Feu de {nom_feu}"] = [[fy0, fx0], [fy1, fx1]]

m.get_root().script.add_child(folium.Element(f"""
  window.addEventListener("load", function () {{
    var zones = {json.dumps(zones)};
    var ctlZones = L.control({{position: 'topleft'}});
    ctlZones.onAdd = function () {{
        var d = L.DomUtil.create('div', 'leaflet-bar');
        d.style.cssText = 'background:#fff;padding:7px 9px;font:12px/1.65 ' +
            'system-ui,-apple-system,sans-serif;color:#0b0b0b;';
        var h = '<div style="font-weight:650;margin-bottom:2px;">Aller à</div>';
        Object.keys(zones).forEach(function (k) {{
            h += '<a href="#" data-z="' + k + '" style="display:block;color:#1c5cab;' +
                 'text-decoration:none;white-space:nowrap;">' + k + '</a>';
        }});
        d.innerHTML = h;
        L.DomEvent.disableClickPropagation(d);
        d.querySelectorAll('a').forEach(function (a) {{
            a.onclick = function (e) {{
                e.preventDefault();
                {m.get_name()}.fitBounds(zones[a.getAttribute('data-z')]);
            }};
        }});
        return d;
    }};
    ctlZones.addTo({m.get_name()});
  }});
"""))

LEGENDE = """
<div style="position:fixed;bottom:38px;left:12px;z-index:9999;background:rgba(255,255,255,.94);
  padding:11px 13px;border-radius:9px;box-shadow:0 1px 8px rgba(0,0,0,.28);
  font:12px/1.45 system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;color:#0b0b0b;">
  <div style="font-weight:650;margin-bottom:6px;">Segment du contrat</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#2a78d6;margin-right:6px;"></span>RP — résidence principale</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#eb6834;margin-right:6px;"></span>RS — résidence secondaire</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#1baf7a;margin-right:6px;"></span>PNO — propriétaire non occupant</div>
  <div style="font-weight:650;margin:9px 0 5px;">Certitude d'impact</div>
  <div>● grand &amp; opaque — dans l'emprise brûlée</div>
  <div>● moyen — à moins de 10 m</div>
  <div>● petit — à moins de 25 m</div>
  <div style="margin-top:7px;"><span style="display:inline-block;width:11px;height:11px;
    background:#1f2937;margin-right:6px;"></span>Bâtiment dans l'emprise brûlée</div>
</div>"""
m.get_root().html.add_child(folium.Element(LEGENDE))


def rendre_carte_autonome(carte: folium.Map) -> str:
    """Rend la carte en HTML sans aucune dépendance CDN.

    folium référence Leaflet, jQuery, Bootstrap et awesome-markers via des <script src>
    et <link href> distants. Sur un poste dont le proxy bloque ces CDN, la page reste
    blanche (incident déjà rencontré sur les cartes de grêle). On retire donc toutes
    les ressources externes et on réinjecte Leaflet — la seule réellement nécessaire —
    depuis `incendie/assets/`.
    """
    import re

    doc = carte.get_root().render()
    if not LEAFLET_EMBARQUE:
        print("⚠️  incendie/assets/leaflet.js absent → la carte dépendra des CDN.")
        return doc

    doc = re.sub(r'<script[^>]+src="https?://[^"]+"[^>]*>\s*</script>', "", doc)
    doc = re.sub(r'<link[^>]+href="https?://[^"]+"[^>]*/?>', "", doc)

    def _lire(nom):
        return (DOSSIER_ASSETS / nom).read_text(encoding="utf-8").replace("</script>", r"<\/script>")

    # jQuery avant Leaflet : folium construit chaque popup avec `$(...)`, et une
    # ReferenceError sur `$` interrompt tout le script — y compris les marqueurs
    # et le sélecteur de couches déclarés plus loin.
    inject = (f"<style>{(DOSSIER_ASSETS / 'leaflet.css').read_text(encoding='utf-8')}</style>\n"
              f"<script>{_lire('jquery.js')}</script>\n"
              f"<script>{_lire('leaflet.js')}</script>\n")
    return doc.replace("</head>", inject + "</head>", 1)


CARTE_HTML = rendre_carte_autonome(m)
print(f"Carte rendue — {len(CARTE_HTML) / 1024:.0f} Ko, "
      f"Leaflet {'embarqué' if LEAFLET_EMBARQUE else 'via CDN'}")
m

## 9. Génération du livrable HTML *one shot*

Un fichier unique : compteurs, méthode, tableaux et carte embarquée. Il s'ouvre
dans n'importe quel navigateur et se transmet tel quel.

In [ ]:
CSS = """
:root{color-scheme:light dark}
*{box-sizing:border-box}
body{margin:0;font:15px/1.6 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;
  background:var(--bg);color:var(--txt)}
:root{--bg:#f4f4f2;--surface:#fcfcfb;--txt:#0b0b0b;--txt2:#52514e;--line:#e0dfda;
  --rp:#2a78d6;--rs:#eb6834;--pno:#1baf7a;--alert:#e34948}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])){
  --bg:#111110;--surface:#1a1a19;--txt:#fff;--txt2:#c3c2b7;--line:#383835;
  --rp:#3987e5;--rs:#d95926;--pno:#199e70;--alert:#e66767}}
:root[data-theme="dark"]{--bg:#111110;--surface:#1a1a19;--txt:#fff;--txt2:#c3c2b7;
  --line:#383835;--rp:#3987e5;--rs:#d95926;--pno:#199e70;--alert:#e66767}
.wrap{max-width:1180px;margin:0 auto;padding:2rem 1.25rem 4rem}
header h1{font-size:1.6rem;margin:0 0 .3rem;letter-spacing:-.01em}
header p{margin:0;color:var(--txt2);font-size:.93rem}
.bandeau{margin:1.25rem 0;padding:.8rem 1rem;border-radius:10px;font-size:.88rem;
  background:#fff7ed;border:1px solid #fed7aa;color:#9a3412}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])) .bandeau{
  background:#2a1a0d;border-color:#7c3d12;color:#fdba74}}
section{background:var(--surface);border:1px solid var(--line);border-radius:14px;
  padding:1.4rem 1.5rem;margin:1.25rem 0}
h2{font-size:1.05rem;margin:0 0 1rem;letter-spacing:-.005em}
h2 .n{color:var(--txt2);font-weight:400;margin-right:.45rem}
.kpis{display:grid;gap:.9rem;grid-template-columns:repeat(auto-fit,minmax(190px,1fr))}
.kpi{border:1px solid var(--line);border-radius:12px;padding:1rem 1.1rem;background:var(--bg)}
.kpi .v{font-size:2.1rem;font-weight:640;line-height:1.05;letter-spacing:-.02em;
  font-variant-numeric:tabular-nums}
.kpi .l{font-size:.8rem;color:var(--txt2);margin-top:.35rem;line-height:1.4}
.kpi.lead .v{color:var(--alert)}
.tbl{overflow-x:auto;-webkit-overflow-scrolling:touch}
table{border-collapse:collapse;width:100%;font-size:.88rem;min-width:520px}
th,td{padding:.55rem .7rem;text-align:right;border-bottom:1px solid var(--line);
  font-variant-numeric:tabular-nums;white-space:nowrap}
th:first-child,td:first-child,th.t,td.t{text-align:left;font-variant-numeric:normal}
thead th{color:var(--txt2);font-weight:600;font-size:.8rem;text-transform:uppercase;
  letter-spacing:.03em;border-bottom:1.5px solid var(--line)}
tbody tr:last-child td{border-bottom:none}
tr.tot td{font-weight:650;border-top:1.5px solid var(--line)}
.pill{display:inline-flex;align-items:center;gap:.4rem;font-weight:600}
.dot{width:9px;height:9px;border-radius:50%;flex:none}
.map{height:660px;border:1px solid var(--line);border-radius:12px;overflow:hidden}
.map iframe{width:100%;height:100%;border:0;display:block}
.notes{font-size:.87rem;color:var(--txt2)}
.notes li{margin-bottom:.5rem}
footer{margin-top:2rem;font-size:.78rem;color:var(--txt2);text-align:center}
"""


def _n(v):
    return f"{int(v):,}".replace(",", " ")


def _tab(df, cls_first=True):
    th = "".join(f"<th{' class=t' if i == 0 and cls_first else ''}>{_html.escape(str(c))}</th>"
                 for i, c in enumerate(df.columns))
    tr = ""
    for _, r in df.iterrows():
        tds = "".join(
            f"<td{' class=t' if i == 0 and cls_first else ''}>"
            f"{_n(v) if isinstance(v, (int, np.integer)) else _html.escape(str(v))}</td>"
            for i, v in enumerate(r))
        tr += f"<tr>{tds}</tr>"
    return f'<div class="tbl"><table><thead><tr>{th}</tr></thead><tbody>{tr}</tbody></table></div>'


# --- KPI par segment -------------------------------------------------------- #
par_seg = (impact.groupby("segment", observed=True)
                 .agg(batis=("bat_id", "nunique"), contrats=("id", "nunique"),
                      societaires=("id_societaire", "nunique"))
                 .reindex(SEGMENTS_CIBLE).fillna(0).astype(int))

kpi_html = f"""
<div class="kpi lead"><div class="v">{_n(KPI['batis_touches'])}</div>
  <div class="l">bâtiments distincts concernés<br>portant un contrat RP / RS / PNO</div></div>
<div class="kpi"><div class="v">{_n(KPI['contrats'])}</div>
  <div class="l">contrats habitation impactés</div></div>
<div class="kpi"><div class="v">{_n(KPI['societaires'])}</div>
  <div class="l">sociétaires distincts à contacter</div></div>
<div class="kpi"><div class="v">{_n(KPI['certains'])}</div>
  <div class="l">dont le point contrat tombe<br><b>dans</b> l'emprise brûlée</div></div>
<div class="kpi"><div class="v">{_n(KPI['en_perimetre'])}</div>
  <div class="l">dans le périmètre du feu mais<br>hors emprise bâtie — exposés</div></div>
"""

seg_rows = "".join(
    f'<tr><td class=t><span class="pill"><span class="dot" style="background:'
    f'{PALETTE_SEGMENT[s]}"></span>{s} — {LIB_SEGMENT[s]}</span></td>'
    f"<td>{_n(par_seg.loc[s, 'batis'])}</td><td>{_n(par_seg.loc[s, 'contrats'])}</td>"
    f"<td>{_n(par_seg.loc[s, 'societaires'])}</td></tr>" for s in SEGMENTS_CIBLE)
seg_rows += (f'<tr class=tot><td class=t>Total RP + RS + PNO</td>'
             f"<td>{_n(KPI['batis_touches'])}</td><td>{_n(KPI['contrats'])}</td>"
             f"<td>{_n(KPI['societaires'])}</td></tr>")

bandeau = ""
if MODE_SOURCE == "simulation":
    bandeau = ('<div class="bandeau"><b>⚠️ Données sociétaires SIMULÉES.</b> '
               "Aucun accès BigQuery ni export local n'a été trouvé : les chiffres et les "
               "points ci-dessous sont un jeu de test destiné à valider la chaîne de "
               "traitement. Relancer le notebook avec accès à "
               "<code>contrat_mgar_gps_iris</code> pour obtenir les résultats réels. "
               "Les données incendie, elles, sont bien les données réelles.</div>")

carte_srcdoc = _html.escape(CARTE_HTML, quote=True)

detail_html = detail.reset_index().rename(columns={"feu_rattache": "Feu"})
sensi_html = sensibilite.rename(columns={
    "seuil_m": "Seuil (m)", "batis_touches": "Bâtiments", "contrats": "Contrats",
    "societaires": "Sociétaires", "retenu": ""})
synth_html = synthese[["feu", "segment", "batis_touches", "contrats", "societaires"]].rename(
    columns={"feu": "Feu", "segment": "Segment", "batis_touches": "Bâtiments",
             "contrats": "Contrats", "societaires": "Sociétaires"})

DOC = f"""<!doctype html>
<html lang="fr"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bâtis sociétaires impactés — incendies Gironde &amp; Biscarrosse</title>
<style>{CSS}</style></head><body><div class="wrap">

<header>
  <h1>Bâtis de sociétaires impactés par les incendies</h1>
  <p>Feux de Gironde et de Biscarrosse — relevés SIG au 26/07/2026 16 h ·
     stock contrats MGAR en cours · analyse du {DATE_ANALYSE}</p>
</header>
{bandeau}

<section>
  <h2><span class="n">1</span>Combien ?</h2>
  <div class="kpis">{kpi_html}</div>
</section>

<section>
  <h2><span class="n">2</span>Répartition par segment</h2>
  <div class="tbl"><table>
    <thead><tr><th class=t>Segment</th><th>Bâtiments concernés</th>
      <th>Contrats</th><th>Sociétaires</th></tr></thead>
    <tbody>{seg_rows}</tbody></table></div>
  <p class="notes" style="margin-top:.9rem">Le total en bâtiments est inférieur à la somme
  des lignes : un même immeuble peut porter plusieurs contrats de segments différents.
  {_n(KPI['hors_cible_impactes'])} contrat(s) impacté(s) relèvent d'autres segments
  (jeune, étudiant, hébergé…) et sortent du périmètre de la demande.</p>
</section>

<section>
  <h2><span class="n">3</span>Où ?</h2>
  <p class="notes" style="margin:-.4rem 0 .9rem">Les deux feux sont distants de 60 km :
  la vue d'ensemble situe les foyers, elle ne permet pas de lire le bâti. Utiliser
  <b>« Aller à »</b> en haut à gauche pour cadrer sur un sinistre — les emprises des
  {_n(len(batis))} bâtiments de l'emprise apparaissent en gris foncé à partir de l'échelle
  du quartier. Le sélecteur en haut à droite active ou masque chaque niveau de
  certitude ; cliquer un point affiche l'adresse et la distance au bâti le plus proche.</p>
  <div class="map"><iframe srcdoc="{carte_srcdoc}" loading="lazy"
    title="Carte des bâtis de sociétaires impactés"></iframe></div>
</section>

<section>
  <h2><span class="n">4</span>Détail par feu</h2>
  {_tab(synth_html)}
  <p class="notes" style="margin:1.1rem 0 .5rem">Ventilation de tous les contrats
  RP / RS / PNO par niveau de certitude :</p>
  {_tab(detail_html)}
</section>

<section>
  <h2><span class="n">5</span>Sensibilité au seuil de distance</h2>
  <p class="notes" style="margin-top:-.4rem">Le géocodage d'une adresse ne tombe pas
  toujours dans l'emprise du bâtiment. Ce tableau montre combien de contrats sont
  comptés selon la tolérance retenue — le seuil de 25 m est celui appliqué ci-dessus.</p>
  {_tab(sensi_html)}
</section>

<section>
  <h2><span class="n">6</span>Méthode et limites</h2>
  <ul class="notes">
    <li><b>Sources.</b> Emprises des bâtiments relevées par la cellule SIG
      ({_n(len(batis))} bâtiments sur les deux feux) et contours des incendies au
      26/07/2026 16 h. Contrats issus de
      <code>contrat_mgar_gps_iris</code> × <code>contrat_mgar</code>
      (<code>tech_date_fin_historisation IS NULL</code>).</li>
    <li><b>Segmentation.</b> <code>code_sous_type</code> : 1–5 → RP, 6 → PNO, 7 → RS.
      Les segments jeune / étudiant / hébergé sont exclus du périmètre de la demande
      mais comptés séparément.</li>
    <li><b>Appariement.</b> Distances calculées en Lambert 93 (EPSG:2154) entre le point
      GPS du contrat et l'emprise du bâtiment relevé le plus proche.</li>
    <li><b>Précision du géocodage.</b> {_n(KPI['suspects'])} contrat(s) impacté(s)
      partagent leurs coordonnées avec au moins 4 autres contrats : probable centroïde
      de commune ou de voie. À vérifier avant tout contact — ces cas sont marqués
      <code>geocodage_suspect</code> dans l'export CSV.</li>
    <li><b>Ce que la donnée ne dit pas.</b> La couche « bâti concerné » recense les
      bâtiments <i>situés dans l'emprise brûlée</i> ; elle ne qualifie pas le degré de
      destruction (totale, partielle, façade). Le chiffre est une population à
      expertiser, pas un nombre de sinistres avérés.</li>
    <li><b>Emprise minimale 50 m².</b> Les annexes plus petites (abris de jardin,
      cabanons) sont absentes de la couche source : un contrat dont seule la dépendance
      a brûlé n'est pas détecté.</li>
  </ul>
</section>

<footer>Généré le {DATE_ANALYSE} — source contrats : {MODE_SOURCE.upper()} ·
Export gestion : <code>{FICHIER_CSV.name}</code></footer>
</div></body></html>"""

FICHIER_HTML.write_text(DOC, encoding="utf-8")
print(f"✅ Livrable écrit : {FICHIER_HTML}  ({FICHIER_HTML.stat().st_size / 1024:.0f} Ko)")

---

### Pour rejouer l'analyse

1. Poste avec accès BigQuery → exécuter tel quel (`Run all`).
2. Poste sans accès → déposer l'export de la requête (cellule 3) dans
   `data_incendie/export_societaires.csv` et relancer.
3. Nouveau feu → ajouter une entrée dans `FEUX` (dossier + motifs de fichiers) ;
   le reste du notebook suit automatiquement.

Le HTML produit est autoportant hors fond de carte : mettre `FOND_DE_CARTE = None`
pour un poste sans accès Internet (les périmètres, bâtis et points restent affichés).